# Mathematical Foundations

Three areas of mathematics carry all the weight in this curriculum:

- **Linear algebra** is what a network *computes* — every layer is a matrix multiply.
- **Calculus** is how it *learns* — backpropagation is the chain rule and nothing else.
- **Probability** is what its outputs *mean*, and where every loss function comes from.

This notebook covers exactly as much of each as the rest of the curriculum needs. It
is the only lesson here that trains no model: everything is verified against a
closed-form answer, against PyTorch, or against a numerical estimate, so you can see
each rule actually hold rather than take it on faith.

Read [Chapter 00 of the book](../book/index.html#ch00) first — this notebook assumes
its explanations and repeats none of them.

**What you will do**

*Part 1 — Linear algebra*
- Dot products as similarity, and matrix multiplication as a grid of dot products
- Shape rules, transposes, and the broadcasting case that silently does the wrong thing

*Part 2 — Calculus*
- Derivatives by finite differences, and gradient checking
- The chain rule, and summing over paths when a variable is used twice
- One gradient-descent step, and what a too-large learning rate does

*Part 3 — Statistics*
- Expectation and variance, and why sigma'(z) IS a Bernoulli variance
- Variance propagation through a deep stack: deriving Xavier init, and watching
  the wrong scale saturate every unit before training starts
- The variance of a dot product, and the sqrt(d_k) in attention
- Standardisation, dropout's 1/(1-p), and the standard error over seeds

*Part 4 — Probability and loss*
- Binary cross-entropy from the Bernoulli likelihood
- Softmax, and the max-subtraction trick that stops it overflowing
- Cross-entropy from the categorical likelihood
- The `ŷ - y` gradient, verified three independent ways
- MSE, MAE, and Huber under an outlier
- Entropy, cross-entropy, KL divergence, and perplexity

---
## A note on this notebook

This lesson is **mostly review**, and it is meant to be skipped around in. It is long
because the later lessons are far more intuitive when this material is fresh — not
because it is new.

If you have a technical background, run the cells, read the printed conclusions, and do
the exercises in the book that are marked **Must do**:

| Must do | why |
|---|---|
| 00.4 · the transpose in the backward pass | shapes force every backward formula |
| 00.7 · the chain rule as backpropagation | this *is* backpropagation |
| 00.12 · variance of a sum, and initialisation | lessons 01, 10, 11 rest on it |
| 00.13 · variance of a dot product, and √dₖ | lessons 06, 07, 09, 10 |
| 00.20 · softmax, cross-entropy, the gradient | every classifier here uses it |
| 00.25 · a network that will not train | the diagnostic loop, end to end |

Six more are marked *Recommended* — each pays off in a specific later lesson. Everything
else is there for the reps, and skipping it costs you nothing but practice.

## Step 1: Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Everything here is small and CPU-bound; no device juggling needed.
torch.manual_seed(0)
np.random.seed(0)

print('torch', torch.__version__)

---
# Part 1 · Linear Algebra

A neural network is a sequence of matrix multiplications with non-linear functions
between them. These operations are not background material — they *are* the forward
pass.

## Step 2: The Dot Product Is a Similarity Score

$$\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i = \|\mathbf{a}\|\|\mathbf{b}\|\cos\theta$$

Large and positive when two vectors point the same way, zero when perpendicular,
negative when opposed. When attention computes $QK^\top$ in lesson 06, every entry
is one of these, asking "how relevant is this position to that one?"

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


pairs = [
    ('same direction', np.array([1.0, 0.0]), np.array([2.0, 0.0])),
    ('45 degrees',     np.array([1.0, 0.0]), np.array([1.0, 1.0])),
    ('perpendicular',  np.array([1.0, 0.0]), np.array([0.0, 1.0])),
    ('opposed',        np.array([1.0, 0.0]), np.array([-1.0, 0.0])),
]

print(f'{"":>16} {"dot":>8} {"cos":>8} {"angle":>8}')
for name, a, b in pairs:
    c = cosine(a, b)
    print(f'{name:>16} {np.dot(a, b):>8.2f} {c:>8.2f} {np.degrees(np.arccos(c)):>7.1f}°')

print()
a = np.array([2.0, 0.0, 1.0])
b = np.array([3.0, 4.0, -2.0])
print(f'a = {a},  b = {b}')
print(f'a · b = (2)(3) + (0)(4) + (1)(-2) = {np.dot(a, b):.0f}')
print()
print('Positive, so the angle is acute: the vectors broadly agree in direction.')

---
## Step 3: Matrix Multiplication, Shapes, and Broadcasting

`(m, n) @ (n, p) -> (m, p)`. The inner dimensions must match, and they vanish.
Entry `(i, j)` of the result is the dot product of row `i` with column `j`.

In [ ]:
A = np.array([[1.0, 2.0, 0.0],
              [0.0, 1.0, 3.0]])          # (2, 3)
B = np.array([[1.0,  0.0],
              [2.0, -1.0],
              [0.0,  1.0]])              # (3, 2)

C = A @ B
print(f'A {A.shape} @ B {B.shape} -> C {C.shape}')
print(C)
print()

# Every entry is one dot product — verify by hand.
for i in range(2):
    for j in range(2):
        terms = ' + '.join(f'({A[i,k]:.0f})({B[k,j]:.0f})' for k in range(3))
        print(f'  C[{i},{j}] = row{i} · col{j} = {terms} = {C[i,j]:.0f}')

print()
print('Order matters — B @ A is a completely different (and here, differently shaped)')
print(f'operation: B {B.shape} @ A {A.shape} -> {(B @ A).shape}')
print()

# The shape errors you will actually hit
print('Illegal shapes raise, and reading the message is a skill:')
try:
    _ = A @ A
except ValueError as e:
    print(f'  A @ A -> {e}')

In [ ]:
# Broadcasting: align from the right; each dim must match or be 1.
X = np.zeros((4, 3))

cases = [
    ('(4,3) + (3,)',   np.ones(3)),
    ('(4,3) + (4,1)',  np.ones((4, 1))),
    ('(4,3) + (1,3)',  np.ones((1, 3))),
    ('(4,3) + (4,)',   np.ones(4)),
]
for label, v in cases:
    try:
        print(f'  {label:<18} -> {(X + v).shape}')
    except ValueError as e:
        print(f'  {label:<18} -> ERROR ({str(e)[:44]}...)')

print()
print('And the case that is legal, silent, and almost never what you meant:')
col = np.array([[1.0], [2.0], [3.0], [4.0]])   # (4, 1)
row = np.array([[10.0, 20.0, 30.0]])           # (1, 3)
print(f'  (4,1) + (1,3) -> {(col + row).shape}   BOTH dimensions stretched')
print(col + row)
print()
print('A (4,1) where you meant (1,4) produces a grid of nonsense rather than an')
print('error. When something is wrong for no visible reason, print your shapes.')

In [ ]:
# A linear layer, by hand and by PyTorch — note the transposed storage.
batch, n_in, n_out = 5, 3, 4
X = torch.randn(batch, n_in)
layer = nn.Linear(n_in, n_out)

manual = X @ layer.weight.T + layer.bias      # PyTorch stores W as (out, in)
auto = layer(X)

print(f'X            {tuple(X.shape)}')
print(f'layer.weight {tuple(layer.weight.shape)}   <- (out, in), the transpose of the maths')
print(f'layer.bias   {tuple(layer.bias.shape)}')
print(f'output       {tuple(auto.shape)}')
print()
print('X @ W.T + b matches nn.Linear:', torch.allclose(manual, auto, atol=1e-6))
print()
print(f'parameters: {n_in}*{n_out} + {n_out} = {n_in*n_out + n_out}')
print('The batch dimension passes through untouched — layers transform features,')
print('never the batch.')

In [ ]:
# Elementwise (Hadamard) vs matrix multiply — completely different operations.
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print(f'a ⊙ b (elementwise) = {a * b}      shape {(a*b).shape}')
print(f'a · b (dot)         = {np.dot(a, b):.0f}                  a scalar')
print()
print('An LSTM forget gate does f ⊙ c: each memory slot is scaled by its own gate')
print('value independently. A dot product would collapse the memory into one number.')

---
### Concept Check: Linear Algebra

1. `A` is `(8, 16)` and `B` is `(16, 4)`. What shape is `A @ B`? Is `B @ A` legal?
2. Why does the backward pass through a weight matrix use `W.T`?
3. You meant to add a bias of shape `(1, 64)` but passed one of shape `(64, 1)` to a
   `(32, 64)` activation. What happens?

In [ ]:
# 1.
# 2.
# 3.

---
# Part 2 · Calculus

Training means adjusting parameters to reduce a loss, and the derivative is the only
thing that says which way. Everything below is in service of one formula:

$$\frac{\partial \mathcal{L}}{\partial \theta} = \text{(product of derivatives along every path from } \theta \text{ to } \mathcal{L})$$

That is backpropagation.

## Step 4: Derivatives, and Gradient Checking

$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$ for small `h`. This is not just a
definition — it is how you verify a hand-written backward pass, and you will use it
on the softmax gradient later in this notebook.

In [ ]:
def numeric_derivative(f, x, h=1e-5):
    """Central difference. More accurate than the forward difference for the same h."""
    return (f(x + h) - f(x - h)) / (2 * h)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))       # used again throughout Part 3


checks = [
    ('x²',       lambda x: x**2,       lambda x: 2*x,                    3.0),
    ('eˣ',       np.exp,               np.exp,                           1.0),
    ('ln x',     np.log,               lambda x: 1/x,                    2.0),
    ('σ(x)',     sigmoid,              lambda x: sigmoid(x)*(1-sigmoid(x)), 0.5),
    ('tanh x',   np.tanh,              lambda x: 1 - np.tanh(x)**2,      0.8),
    ('(3x+1)²',  lambda x: (3*x+1)**2, lambda x: 6*(3*x+1),              2.0),
]

print(f'{"function":>10} {"at x":>6} {"analytic":>12} {"numeric":>12} {"agree":>7}')
for name, f, df, x in checks:
    a, n = df(x), numeric_derivative(f, x)
    print(f'{name:>10} {x:>6.1f} {a:>12.6f} {n:>12.6f} {str(np.isclose(a, n)):>7}')

print()
print('The last row is the chain rule: d/dx (3x+1)² = 2(3x+1) · 3 = 6(3x+1).')
print('Multiply the derivatives of the composed functions.')

---
## Step 5: The Multivariable Chain Rule — Sum Over Paths

When a variable influences the loss by more than one route, the total derivative is
the **sum over all paths**. This is exactly why PyTorch *accumulates* into `.grad`
rather than overwriting it — and therefore why `zero_grad()` is mandatory.

In [ ]:
# u = w², v = u + w, L = 3v.  w reaches L by two paths.
w0 = 2.0

def L_of(w):
    u = w**2
    v = u + w
    return 3*v

# analytic, path by path
dL_dv = 3.0
dv_du, du_dw = 1.0, 2*w0        # path 1: w -> u -> v -> L
dv_dw = 1.0                     # path 2: w -> v -> L

path1 = du_dw * dv_du * dL_dv
path2 = dv_dw * dL_dv
analytic = path1 + path2

print(f'path 1  (w -> u -> v -> L): {du_dw} x {dv_du} x {dL_dv} = {path1}')
print(f'path 2  (w -> v -> L):          {dv_dw} x {dL_dv} = {path2}')
print(f'total   dL/dw = {analytic}')
print()
print(f'numeric check: {numeric_derivative(L_of, w0):.4f}')

# and autograd agrees
wt = torch.tensor(w0, requires_grad=True)
(3 * (wt**2 + wt)).backward()
print(f'autograd     : {wt.grad.item():.4f}')
print()

# Now the bug this behaviour causes.
wt2 = torch.tensor(w0, requires_grad=True)
(3 * (wt2**2 + wt2)).backward()
print(f'after one backward()  : w.grad = {wt2.grad.item():.1f}')
(3 * (wt2**2 + wt2)).backward()
print(f'after two backwards   : w.grad = {wt2.grad.item():.1f}   <- the missing zero_grad() bug')
print()
print('Nothing crashes. 30 is the derivative of nothing in particular, and a model')
print('trained this way descends steadily towards the wrong thing.')

---
## Step 6: Gradient Descent, and What the Learning Rate Does

$$\\theta \\leftarrow \\theta - \\eta \\frac{\\partial \\mathcal{L}}{\\partial \\theta}$$

In [ ]:
def descend(lr, steps=12, w=1.0):
    """Minimise L(w) = (w - 4)², whose minimum is obviously at w = 4."""
    path = [w]
    for _ in range(steps):
        grad = 2 * (w - 4)
        w = w - lr * grad
        path.append(w)
    return path


fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ws = np.linspace(-6, 14, 300)
for lr, color in [(0.1, '#00b894'), (0.5, '#0984e3'), (0.9, '#fdcb6e'), (1.05, '#d63031')]:
    path = descend(lr)
    ax[0].plot(path, marker='o', ms=3, color=color, label=f'lr={lr}')
    print(f'lr={lr:<5} final w = {path[-1]:>12.4f}   loss = {(path[-1]-4)**2:>14.6f}')

ax[0].axhline(4, color='#636e72', ls='--', lw=1, label='optimum')
ax[0].set_ylim(-8, 16)
ax[0].set_xlabel('step'); ax[0].set_ylabel('w')
ax[0].set_title('Learning rate decides everything')
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)

ax[1].plot(ws, (ws - 4)**2, color='#b2bec3', lw=2)
p = descend(0.1)
ax[1].plot(p, [(x-4)**2 for x in p], 'o-', color='#00b894', ms=4, label='lr=0.1')
p = descend(1.05, steps=6)
ax[1].plot(p, [(x-4)**2 for x in p], 'o-', color='#d63031', ms=4, label='lr=1.05')
ax[1].set_ylim(0, 60); ax[1].set_xlabel('w'); ax[1].set_ylabel('L(w)')
ax[1].set_title('Descending, and diverging')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print()
print('At lr=1.05 each step overshoots by more than it corrects, so the loss grows')
print('without bound. That is what a nan loss five steps into training usually means,')
print('and lowering the learning rate is usually the whole fix.')

---
### Concept Check: Calculus

1. Your analytic gradient and a finite-difference estimate disagree by 30%. Which do
   you trust, and what do you do?
2. Why must `zero_grad()` be called every iteration, given that summing over paths is
   the mathematically correct behaviour?
3. A network has 10 sigmoid layers. The derivative of sigmoid peaks at 0.25. What is
   the best possible gradient magnitude reaching layer 1, relative to the loss?

In [ ]:
# 1.
# 2.
# 3.

---
# Part 3 · Statistics

Initialisation, normalisation, dropout, and the $\sqrt{d_k}$ in attention are not
folklore — each is a **variance argument**. This part computes them.

Two rules do all the work:

$$\operatorname{Var}(aX) = a^2\operatorname{Var}(X)
\qquad\qquad
\operatorname{Var}(X + Y) = \operatorname{Var}(X) + \operatorname{Var}(Y)\ \text{(independent)}$$

## Step 7: Expectation and Variance, and a Coincidence That Is Not One

In [ ]:
def moments(values, probs):
    """E[X] and Var(X) for a discrete distribution, from the definition."""
    values, probs = np.asarray(values, float), np.asarray(probs, float)
    # TODO: E[X] = sum of value x probability
    ex = ...
    # TODO: E[X^2], the same sum with the values squared
    ex2 = ...
    # TODO: return E[X] and Var(X) = E[X^2] - E[X]^2
    return ...


die_v, die_p = [1, 2, 3, 4, 5, 6], [1/6] * 6
ex, var = moments(die_v, die_p)
print(f'fair die   : E[X] = {ex:.4f}   Var(X) = {var:.4f}   SD = {np.sqrt(var):.4f}')

for p in (0.1, 0.3, 0.5, 0.7, 0.9):
    ex, var = moments([0, 1], [1 - p, p])
    print(f'Bernoulli p={p:.1f}: E[X] = {ex:.2f}   Var(X) = {var:.4f}   '
          f'p(1-p) = {p*(1-p):.4f}')

print()
print('Var of a Bernoulli is p(1-p), maximised at p = 0.5 where it equals 0.25.')
print()

# The coincidence: sigmoid's derivative IS a Bernoulli variance.
z = np.linspace(-6, 6, 7)
print(f'{"z":>6} {"sigma(z)":>10} {"sigma\'(z)":>11} {"p(1-p)":>10}')
for zz in z:
    p = sigmoid(zz)
    print(f'{zz:>6.1f} {p:>10.4f} {p*(1-p):>11.4f} {p*(1-p):>10.4f}')

print()
print('These are the same column, because a sigmoid output IS a Bernoulli parameter')
print('and its derivative IS that Bernoulli\'s variance. So a confident unit and an')
print('unlearnable unit are the same condition, read two ways.')

---
## Step 8: Variance Propagation, and Where Xavier Initialisation Comes From

A unit computes $z = \sum_{i=1}^{n} w_ix_i$. With inputs of variance 1 and weights
independent with variance $\sigma_w^2$, the two rules above give

$$\operatorname{Var}(z) = n\,\sigma_w^2$$

so keeping the signal at the same scale across a layer requires
$\sigma_w = 1/\sqrt{n}$. That is Xavier/LeCun initialisation, in one line.

The cell below propagates a signal through 20 layers at three different
initialisations and watches what happens.

In [ ]:
def propagate(n_layers=20, width=256, sigma_w=None, activation=None, seed=0):
    """Push a unit-variance signal through a stack and record the variance."""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(4096, width, generator=g)          # variance 1 by construction
    variances = [x.var().item()]
    for _ in range(n_layers):
        # TODO: draw a (width, width) weight matrix with standard deviation
        # sigma_w, then push x through it. Var(z) = n * sigma_w^2 predicts
        # what you are about to see.
        W = ...
        x = ...
        if activation is not None:
            x = activation(x)
        variances.append(x.var().item())
    return variances


width = 256
xavier = 1 / np.sqrt(width)

runs = {
    f'sigma_w = 1.0        (too big)': propagate(sigma_w=1.0, width=width),
    f'sigma_w = {xavier:.4f}  (Xavier 1/sqrt(n))': propagate(sigma_w=xavier, width=width),
    f'sigma_w = 0.01       (too small)': propagate(sigma_w=0.01, width=width),
}

print(f'{"layer":>6}' + ''.join(f'{k.split()[2]:>14}' for k in runs))
for layer in (0, 1, 2, 5, 10, 20):
    row = f'{layer:>6}'
    for v in runs.values():
        row += f'{v[layer]:>14.3e}'
    print(row)

print()
for name, v in runs.items():
    print(f'{name:<34} variance after 20 layers: {v[-1]:.3e}')
print()
print('Only the middle one is usable. The others differ from it by tens of orders of')
print('magnitude — and the ONLY difference between the three runs is one scalar.')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.7))

for name, v in runs.items():
    ax[0].semilogy(np.array(v) + 1e-30, marker='o', ms=3, label=name.split('(')[1].rstrip(')'))
ax[0].axhline(1.0, color='#636e72', ls='--', lw=1)
ax[0].set_xlabel('layer'); ax[0].set_ylabel('activation variance (log scale)')
ax[0].set_title('Variance propagation through 20 layers')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

# What the wrong scale does to a sigmoid layer, concretely.
g = torch.Generator().manual_seed(0)
x = torch.randn(4096, width, generator=g)
for sig, color, lab in ((xavier, '#00b894', f'1/sqrt(n) = {xavier:.3f}'),
                        (1.0, '#d63031', 'sigma_w = 1.0')):
    z = x @ (torch.randn(width, width, generator=g) * sig)
    ax[1].hist(z.flatten().numpy(), bins=120, alpha=0.55, color=color,
               density=True, label=f'{lab}  (SD {z.std():.2f})')
ax[1].set_xlim(-40, 40)
ax[1].set_xlabel('pre-activation z'); ax[1].set_ylabel('density')
ax[1].set_title('Where the pre-activations land')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# and the gradient consequence
for zz in (1.0, 10.0, 30.0):
    p = sigmoid(zz)
    print(f'z = {zz:5.1f}:  sigma(z) = {p:.10f}   sigma\'(z) = {p*(1-p):.3e}')
print()
print('At sigma_w = 1 with n = 256, pre-activations have SD 16, so most units sit')
print('past |z| = 10 where the derivative is under 5e-5. Every unit is saturated')
print('BEFORE the first gradient step, and the network never trains. Nothing in the')
print('code is wrong — only a factor of sqrt(n).')

In [ ]:
# He initialisation: ReLU discards the negative half, roughly halving the variance.
relu = lambda t: torch.relu(t)

for label, sig in (('Xavier  1/sqrt(n)', xavier), ('He  sqrt(2/n)', np.sqrt(2 / width))):
    v = propagate(n_layers=10, width=width, sigma_w=sig, activation=relu)
    print(f'{label:<20} variance by layer: ' +
          ' '.join(f'{x:.3f}' for x in v[:6]) + ' ...  final ' + f'{v[-1]:.4f}')

print()
print('With ReLU, Xavier decays by about half per layer (0.5^10 ~ 1e-3 after ten).')
print('He initialisation doubles the weight variance to compensate, and holds steady.')
print('That factor of sqrt(2) is the entire difference between the two schemes.')

---
## Step 9: The Variance of a Dot Product, and the $\sqrt{d_k}$ in Attention

For $q\cdot k = \sum_i q_ik_i$ with independent unit-variance components:

$$\mathbb{E}[q_ik_i] = 0
\qquad
\operatorname{Var}(q_ik_i) = \mathbb{E}[q_i^2]\mathbb{E}[k_i^2] = 1
\qquad
\operatorname{Var}(q\cdot k) = d_k$$

So attention scores have standard deviation $\sqrt{d_k}$ — they grow with model
width. Since softmax cares about *absolute* differences between logits, widening the
model would saturate every attention layer. Dividing by $\sqrt{d_k}$ fixes it.

In [ ]:
g = torch.Generator().manual_seed(0)

print(f'{"d_k":>6} {"measured SD":>13} {"sqrt(d_k)":>11} {"max softmax":>13} {"max p(1-p)":>12}')
for d_k in (4, 16, 64, 256, 1024):
    q = torch.randn(20000, d_k, generator=g)
    k = torch.randn(20000, d_k, generator=g)
    scores = (q * k).sum(dim=1)

    # a representative row of three scores at 0 and +/- 1 SD
    sd = scores.std().item()
    row = torch.tensor([0.0, sd, -sd])
    p = torch.softmax(row, dim=0)
    pmax = p.max().item()

    print(f'{d_k:>6} {sd:>13.3f} {np.sqrt(d_k):>11.3f} {pmax:>13.6f} {pmax*(1-pmax):>12.2e}')

print()
print('The measured SD tracks sqrt(d_k) exactly, as predicted. And the softmax')
print('Jacobian term p(1-p) — the same Bernoulli variance from Step 7 — collapses')
print('as d_k grows: at d_k=1024 the attention weights receive essentially no')
print('gradient at all.')
print()

# now with the scaling applied
print('with the 1/sqrt(d_k) scaling applied:')
print(f'{"d_k":>6} {"scaled SD":>11} {"max softmax":>13} {"max p(1-p)":>12}')
for d_k in (4, 16, 64, 256, 1024):
    q = torch.randn(20000, d_k, generator=g)
    k = torch.randn(20000, d_k, generator=g)
    # TODO: the same dot product, divided by sqrt(d_k)
    scores = ...
    sd = scores.std().item()
    row = torch.tensor([0.0, sd, -sd])
    p = torch.softmax(row, dim=0)
    pmax = p.max().item()
    print(f'{d_k:>6} {sd:>11.3f} {pmax:>13.6f} {pmax*(1-pmax):>12.4f}')

print()
print('Constant at every width. That is the whole point: score magnitude becomes')
print('independent of dimension, so making the model wider cannot break it.')

---
## Step 10: Standardisation, Dropout, and the Standard Error

Three more variance arguments, each behind a line of code you have already written
or are about to.

In [ ]:
# --- standardisation is what layer norm does, per token ------------------------
x = torch.tensor([2., 4., 4., 4., 5., 5., 7., 9.])
mu, sd = x.mean(), x.std(unbiased=False)
z = (x - mu) / sd

print(f'x        = {x.numpy()}')
print(f'mean     = {mu:.4f}    sd = {sd:.4f}')
print(f'z        = {z.numpy()}')
print(f'z mean   = {z.mean():.6f}   z var = {z.var(unbiased=False):.6f}')

ln = nn.LayerNorm(8, elementwise_affine=False)
print(f'nn.LayerNorm agrees: {torch.allclose(ln(x), z, atol=1e-5)}')
print()

# with learned gamma and beta the network can undo the normalisation entirely
gamma, beta = 2.0, 1.0
y = gamma * z + beta
print(f'with gamma=2, beta=1:  mean = {y.mean():.4f}  var = {y.var(unbiased=False):.4f}')
print('The a^2 rule: scaling by 2 multiplies the variance by 4.')

In [ ]:
# --- dropout: the mean is preserved, the variance is not -----------------------
torch.manual_seed(0)
x = torch.ones(200000)

print(f'{"p":>5} {"E[y] naive":>12} {"E[y] inverted":>15} {"Var(y)":>10} {"predicted":>11}')
for p in (0.1, 0.5, 0.9):
    keep = (torch.rand_like(x) > p).float()
    naive = x * keep                      # what dropout would do without rescaling
    inverted = x * keep / (1 - p)         # what it actually does
    predicted_var = (1.0 ** 2) * p / (1 - p)
    print(f'{p:>5.1f} {naive.mean():>12.4f} {inverted.mean():>15.4f} '
          f'{inverted.var():>10.4f} {predicted_var:>11.4f}')

print()
print('Naive dropout shrinks the mean to (1-p)x, so every downstream layer sees a')
print('different input scale in training than at test time. Dividing by (1-p) fixes')
print('the mean exactly — but Var(y) = x^2 p/(1-p) grows without bound, which is why')
print('dropout above ~0.5 destabilises training rather than merely slowing it.')

In [ ]:
# --- how many seeds before you can claim anything ------------------------------
accs = np.array([0.812, 0.789, 0.834])

mean = accs.mean()
s = accs.std(ddof=1)                     # sample sd, dividing by n-1
se = s / np.sqrt(len(accs))

print(f'accuracies : {accs}')
print(f'mean       : {mean:.4f}')
print(f's (n-1)    : {s:.4f}')
print(f'std error  : {se:.4f}')
print(f'~95% CI    : [{mean - 2*se:.4f}, {mean + 2*se:.4f}]')
print()
print(f'A competing model scores 0.820 on ONE run.')
print(f'  inside the interval? {mean - 2*se < 0.820 < mean + 2*se}')
print('  -> the data cannot distinguish the two architectures.')
print()

print(f'{"seeds":>6} {"std error":>11}   (s held fixed)')
for n in (1, 3, 12, 48):
    print(f'{n:>6} {s/np.sqrt(n):>11.5f}')
print()
print('SE falls as 1/sqrt(n): halving your uncertainty costs 4x the runs. That is')
print('why the right response to a noisy result is usually a bigger effect, not')
print('more seeds — and why reporting a mean with no spread is not a result.')

---
### Concept Check: Statistics

1. You widen a transformer from `d_k=64` to `d_k=256` and it gets *worse*, with
   attention entropy collapsing. What happened, and what is the one-line fix?
2. A colleague initialises a 30-layer network with `torch.randn(n, n)` and reports
   that the loss "starts high and never moves". Diagnose it without seeing the code.
3. Your model scores 84.1% and a colleague's scores 84.9%, each on one seed. What
   can you conclude, and what would you need to conclude something?

In [ ]:
# 1.
# 2.
# 3.

---
# Part 4 · Probability and Loss

---
## Key Concept: From Likelihood to Loss

For a dataset of `N` independent examples, the likelihood of the parameters is the
**product** of the per-example probabilities:

$$\mathcal{l}(\theta) = \prod_{i=1}^{N} P(y_i \mid x_i, \theta)$$

We do not use that product directly, for two reasons:

1. **It underflows.** 64 examples at probability 0.1 each gives `1e-64`; float32
   bottoms out around `1e-38`. The likelihood becomes exactly zero, and so does its
   gradient.
2. **Derivatives of products are miserable.** The product rule compounds across all
   `N` terms.

Taking a logarithm fixes both. It is monotonic, so the maximiser is unchanged, and
it turns the product into a sum. Flip the sign to make it a minimisation and divide
by `N` to make it comparable across batch sizes:

$$\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N}\log P(y_i \mid x_i, \theta)$$

**Every loss below is that formula with a different distribution substituted in.**

## Step 11: The Underflow That Motivates the Log

In [ ]:
# 64 examples, each assigned probability 0.1 by the model.
probs = np.full(64, 0.1, dtype=np.float32)

product = np.prod(probs)                 # the raw likelihood
log_sum = np.sum(np.log(probs))          # the log-likelihood

print(f'raw product        : {product}')          # underflows to 0.0
print(f'sum of logs        : {log_sum:.4f}')      # perfectly well behaved
print(f'exp(sum of logs)   : {np.exp(log_sum)}')  # 0.0 again — the value is genuinely tiny
print()
print(f'product == 0.0 ?   : {product == 0.0}')
print('A zero likelihood has zero gradient. Training would stop before it started.')

---
## Step 12: Bernoulli → Binary Cross-Entropy

A single yes/no outcome with probability `p` of being 1:

$$P(y \mid p) = p^{y}(1-p)^{1-y}$$

The exponents act as an `if` statement written as an expression — when `y=1` it
reads `p`, when `y=0` it reads `1-p`. Substituting into the negative
log-likelihood gives

$$\mathcal{L} = -\frac{1}{N}\sum_i\Big[y_i\log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\Big]$$

**TODO:** implement `bce` from the formula, then check it against `nn.BCELoss`.
Clip the predictions before taking the log, or a prediction of exactly 0 or 1 will
produce `-inf`.

In [ ]:
def bce(y_true, y_pred, eps=1e-7):
    """Binary cross-entropy, averaged over the batch.

    y_true : (N,) array of 0/1 labels
    y_pred : (N,) array of predicted probabilities in (0, 1)
    """
    # TODO: clip y_pred to [eps, 1 - eps] so log() never sees 0
    y_pred = ...
    # TODO: return the mean of -(y*log(ŷ) + (1-y)*log(1-ŷ))
    return ...


y_true = np.array([1.0, 0.0, 1.0, 0.0, 1.0])
y_pred = np.array([0.9, 0.2, 0.4, 0.1, 0.95])

ours = bce(y_true, y_pred)
theirs = nn.BCELoss()(torch.tensor(y_pred), torch.tensor(y_true)).item()

print(f'our bce      : {ours:.6f}')
print(f'nn.BCELoss   : {theirs:.6f}')
print(f'agree        : {np.isclose(ours, theirs)}')
print()

# Per-example contributions: which one dominates the batch?
per = -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
for t, p, l in zip(y_true, y_pred, per):
    print(f'  y={t:.0f}  ŷ={p:.2f}   loss={l:.4f}   ({100*l/per.sum():.0f}% of the total)')

Notice how unevenly the batch loss is distributed. The one example on the wrong side
of 0.5 contributes far more than the four confident-and-correct ones combined. That
asymmetry is the log at work, and it is why a handful of hard examples can dominate
a gradient step.

---
## Step 13: Softmax, and Why It Overflows

$$\hat{y}_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

Exponentiating forces positivity; dividing by the sum forces normalisation. Written
naively it also overflows: `exp(800)` is `inf` in float64, and `inf/inf` is `nan`.

Softmax is invariant to adding a constant to every logit, because $e^c$ factors out
of the numerator and denominator alike. So subtract the maximum first — the largest
exponent becomes $e^0 = 1$, and overflow is impossible.

**TODO:** implement both versions and compare them on a large logit.

In [ ]:
def softmax_naive(z):
    """Straight from the formula. Overflows for large logits."""
    # TODO: exponentiate z, then divide by the sum
    e = ...
    return ...


def softmax_stable(z):
    """Subtract the max first. Mathematically identical, numerically safe."""
    # TODO: exponentiate (z - z.max()), then divide by the sum
    e = ...
    return ...


small = np.array([2.0, 1.0, 0.0])
large = np.array([800.0, 799.0, 798.0])   # same differences, huge magnitude

print('small logits')
print(f'  naive  : {softmax_naive(small)}')
print(f'  stable : {softmax_stable(small)}')
print()
print('large logits (identical differences, so the answer should be identical)')
with np.errstate(over='ignore', invalid='ignore'):
    print(f'  naive  : {softmax_naive(large)}')
print(f'  stable : {softmax_stable(large)}')
print()
print(f'torch    : {F.softmax(torch.tensor(large), dim=0).numpy()}')

---
## Step 14: Categorical → Cross-Entropy

$$P(y \mid p) = \prod_k p_k^{y_k} \qquad\Longrightarrow\qquad
\mathcal{L} = -\frac{1}{N}\sum_i \log \hat{y}_{i, c_i}$$

Because `y` is one-hot, the sum over classes collapses to a single term: the log of
the probability assigned to the correct class. This is why
`nn.CrossEntropyLoss` takes **integer class indices** rather than one-hot vectors —
the one-hot was only ever a device for selecting one term.

Note also that `nn.CrossEntropyLoss` expects **raw logits**, not probabilities. It
applies the log-softmax internally, in a fused and numerically stable form.

In [ ]:
def cross_entropy(logits, targets):
    """Cross-entropy from logits.

    logits  : (N, K) raw scores
    targets : (N,)   integer class indices
    """
    # TODO: compute log-probabilities stably.
    #   log_softmax(z) = z - m - log(sum(exp(z - m)))   where m = row max
    m = ...
    log_probs = ...
    # TODO: average -log_probs at the correct class of each row.
    #   Hint: log_probs[np.arange(len(targets)), targets]
    return ...


logits = np.array([[2.0, 1.0, 0.0],
                   [0.5, 2.5, 0.1],
                   [1.0, 1.0, 3.0]])
targets = np.array([1, 1, 0])     # note: rows 1 and 3 are wrong, row 2 is right

ours = cross_entropy(logits, targets)
theirs = nn.CrossEntropyLoss()(torch.tensor(logits), torch.tensor(targets)).item()

print(f'our cross_entropy   : {ours:.6f}')
print(f'nn.CrossEntropyLoss : {theirs:.6f}')
print(f'agree               : {np.isclose(ours, theirs)}')
print()
print(f'uniform-guess loss for 3 classes = -ln(1/3) = {-np.log(1/3):.4f}')
print('A loss above that means the model is doing worse than guessing.')

---
## Key Concept: The Gradient That Makes Classification Work

Differentiate softmax-then-cross-entropy with respect to the **logits** and almost
everything cancels:

$$\frac{\partial \mathcal{L}}{\partial z_k} = \hat{y}_k - y_k$$

Predicted probability minus true probability. No saturation factor, no attenuation.
Confidently wrong gives a large gradient; nearly right gives a small one.

This cancellation is the reason you should pass **logits** to
`nn.CrossEntropyLoss` and `nn.BCEWithLogitsLoss` rather than applying softmax or
sigmoid yourself. Doing it yourself still runs, still trains, and is measurably worse.

Below we verify the formula three independent ways: analytically, by finite
differences, and with autograd.

In [ ]:
z = np.array([2.0, 1.0, 0.0])
c = 1                                  # true class
y_onehot = np.zeros(3); y_onehot[c] = 1

# --- 1. analytic: ŷ - y ---------------------------------------------------------
p = softmax_stable(z)
analytic = p - y_onehot

# --- 2. finite differences: (L(z+h) - L(z-h)) / 2h -------------------------------
def loss_at(zv):
    return -np.log(softmax_stable(zv)[c])

h = 1e-6
numeric = np.zeros(3)
for k in range(3):
    zp, zm = z.copy(), z.copy()
    zp[k] += h; zm[k] -= h
    numeric[k] = (loss_at(zp) - loss_at(zm)) / (2 * h)

# --- 3. autograd ----------------------------------------------------------------
zt = torch.tensor(z, requires_grad=True)
nn.CrossEntropyLoss()(zt.unsqueeze(0), torch.tensor([c])).backward()
autograd = zt.grad.numpy()

print(f'softmax(z)          : {np.round(p, 4)}')
print(f'analytic  (ŷ - y)   : {np.round(analytic, 6)}')
print(f'finite differences  : {np.round(numeric, 6)}')
print(f'autograd            : {np.round(autograd, 6)}')
print()
print(f'all three agree     : {np.allclose(analytic, numeric, atol=1e-5) and np.allclose(analytic, autograd)}')
print(f'gradients sum to    : {analytic.sum():.2e}')
print()
print('They sum to zero because softmax outputs are constrained to sum to 1:')
print('adding a constant to every logit changes nothing, so the gradient can have')
print('no component in that direction. Only relative logits are learnable.')

---
### Concept Check: Likelihood and Cross-Entropy

Answer these in the cell below **before** moving on.

1. Why do we take the logarithm of the likelihood? Give both reasons.
2. `nn.CrossEntropyLoss` takes integer targets rather than one-hot vectors. What in
   the derivation makes that possible?
3. The three gradients above sum to zero. What does that tell you about what a
   classifier can and cannot learn?

In [ ]:
# 1.
# 2.
# 3.

---
## Step 15: Gaussian → MSE, and Why It Fails on Classification

Substituting the Gaussian density into the negative log-likelihood gives

$$-\log P(y \mid \mu, \sigma^2) = \frac{(y-\mu)^2}{2\sigma^2} + \tfrac{1}{2}\log(2\pi\sigma^2)$$

The second term does not involve $\mu$, so it has zero gradient and can be dropped;
a fixed $\sigma$ is a constant scale absorbed into the learning rate. What is left
is MSE. **So MSE is not assumption-free** — it encodes a belief that your targets
are the truth plus symmetric, constant-width Gaussian noise.

Used for classification behind a sigmoid, it has a specific and damaging failure:

$$\frac{\partial}{\partial z}(\hat{y}-y)^2 = 2(\hat{y}-y)\,\sigma'(z)$$

and $\sigma'(z) \to 0$ exactly when the model is most confidently wrong.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


z = np.linspace(-10, 10, 400)
y_true = 1.0                       # the truth is 1 throughout
p = sigmoid(z)

grad_mse = 2 * (p - y_true) * p * (1 - p)   # chain rule through the sigmoid
grad_bce = p - y_true                       # the cancellation

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].plot(z, np.abs(grad_mse), color='#d63031', lw=2, label='MSE + sigmoid')
ax[0].plot(z, np.abs(grad_bce), color='#0984e3', lw=2, label='BCE + sigmoid')
ax[0].axvline(-6, color='#636e72', ls='--', lw=1)
ax[0].annotate('confidently WRONG\n(needs a big gradient)', xy=(-6, 0.5),
               xytext=(-9.5, 0.72), fontsize=8, color='#636e72')
ax[0].set_xlabel('logit z'); ax[0].set_ylabel('|gradient w.r.t. z|')
ax[0].set_title('Gradient magnitude when the truth is 1')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].semilogy(z, np.abs(grad_mse) + 1e-12, color='#d63031', lw=2, label='MSE + sigmoid')
ax[1].semilogy(z, np.abs(grad_bce) + 1e-12, color='#0984e3', lw=2, label='BCE + sigmoid')
ax[1].set_xlabel('logit z'); ax[1].set_ylabel('|gradient| (log scale)')
ax[1].set_title('Same thing, log scale')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

for zz in (-8.0, -4.0, 0.0):
    pp = sigmoid(zz)
    print(f'z={zz:5.1f}  ŷ={pp:.6f}   MSE grad={abs(2*(pp-1)*pp*(1-pp)):.3e}   '
          f'BCE grad={abs(pp-1):.3e}')
print()
print('At z=-8 the model is as wrong as it can be, and MSE gives it a gradient of')
print('~7e-4 while BCE gives ~1.0 — a factor of 1400. MSE stalls precisely where')
print('learning matters most.')

---
## Step 16: MSE vs MAE vs Huber Under an Outlier

Different distributions give different regression losses:

| Loss | Distribution | Gradient w.r.t. error `e` |
|------|--------------|---------------------------|
| MSE  | Gaussian     | `2e` — grows without bound |
| MAE  | Laplace      | `sign(e)` — constant magnitude |
| Huber| hybrid       | `e` near 0, clipped to `±δ` in the tails |

The consequence is entirely practical: one mislabelled point can dominate an MSE
batch and barely register under MAE.

In [ ]:
# Nine clean points near 0 and one badly mislabelled point at 10.
errors = np.array([0.1, -0.2, 0.05, 0.3, -0.1, 0.2, -0.05, 0.15, -0.25, 10.0])
delta = 1.0

mse_grads = 2 * errors
mae_grads = np.sign(errors)
hub_grads = np.where(np.abs(errors) <= delta, errors, delta * np.sign(errors))

print(f'{"error":>8} {"MSE grad":>10} {"MAE grad":>10} {"Huber grad":>11}')
for e, m, a, h in zip(errors, mse_grads, mae_grads, hub_grads):
    tag = '  <-- outlier' if abs(e) > 5 else ''
    print(f'{e:8.2f} {m:10.2f} {a:10.2f} {h:11.2f}{tag}')

print()
for name, g in (('MSE', mse_grads), ('MAE', mae_grads), ('Huber', hub_grads)):
    share = abs(g[-1]) / np.abs(g).sum()
    print(f'{name:>5}: the single outlier supplies {100*share:5.1f}% of the total gradient')

print()
print('Under MSE one bad label out of ten steers the batch. Under MAE it gets one')
print('vote like everything else. Huber gives you MSE smoothness near zero and MAE')
print('robustness in the tail.')

---
## Step 17: Entropy, Cross-Entropy, and KL Divergence

A second derivation of the same loss, from information theory.

- **Surprisal** of an event of probability `p` is `-log p` — how many bits an
  optimal code spends on it.
- **Entropy** `H(p) = -Σ p log p` is the expected surprisal under the truth: the
  irreducible cost.
- **Cross-entropy** `H(p,q) = -Σ p log q` is what you pay encoding reality `p`
  using beliefs `q`.
- **KL divergence** `D(p‖q) = H(p,q) - H(p) ≥ 0` is the pure penalty for being wrong.

Since `H(p)` depends only on the data, minimising cross-entropy **is** minimising
KL divergence from the truth. The maximum-likelihood story and the information
theory story are the same story.

In [ ]:
def entropy(p):
    p = np.asarray(p, dtype=float)
    return -np.sum(p * np.log2(p + 1e-12))


def cross_entropy_pq(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    return -np.sum(p * np.log2(q + 1e-12))


def kl(p, q):
    return cross_entropy_pq(p, q) - entropy(p)


p_true = np.array([0.5, 0.5])

for q in ([0.5, 0.5], [0.25, 0.75], [0.1, 0.9], [0.01, 0.99]):
    q = np.array(q)
    print(f'q = {q}   H(p,q) = {cross_entropy_pq(p_true, q):.4f} bits   '
          f'KL(p||q) = {kl(p_true, q):.4f} bits')

print()
print(f'H(p) = {entropy(p_true):.4f} bits  <- the floor; no model beats this')
print()

# KL is asymmetric — this is a feature, not a defect.
a, b = np.array([0.5, 0.5]), np.array([0.25, 0.75])
print(f'KL(a||b) = {kl(a, b):.4f}    KL(b||a) = {kl(b, a):.4f}   equal? {np.isclose(kl(a,b), kl(b,a))}')
print()
print('Each averages the discrepancy under a different distribution, so "how')
print('surprised will I be" has a different answer depending on who is expecting.')

### Perplexity

For a language model, **perplexity** is `exp(cross-entropy in nats)`. It has a
direct reading: a perplexity of 40 means the model is, on average, as uncertain as
if it were choosing uniformly among 40 words. Loss curves are hard to interpret;
perplexity is not.

In [ ]:
for loss_nats in (0.0, 0.7, 1.8, 3.0, 4.6, np.log(50000)):
    print(f'cross-entropy {loss_nats:6.3f} nats  ->  perplexity {np.exp(loss_nats):10.1f}')

print()
print('The last row is a model that has learned nothing about a 50k vocabulary.')
print('Chapter 10 gets to roughly 1.8 nats on a character-level task.')

---
## Step 18: Choosing a Loss

| Task | Final layer | Loss | PyTorch |
|------|-------------|------|---------|
| Binary classification | 1 logit | BCE | `BCEWithLogitsLoss` |
| Multi-class, one label | K logits | cross-entropy | `CrossEntropyLoss` |
| Multi-label (tags) | K logits | BCE per class | `BCEWithLogitsLoss` |
| Regression, clean | 1 linear | MSE | `MSELoss` |
| Regression, outliers | 1 linear | MAE / Huber | `L1Loss`, `SmoothL1Loss` |
| Next-token prediction | V logits | cross-entropy | `CrossEntropyLoss` |

Three rules that cause more bugs than anything else:

1. **Pass logits, not probabilities.** The fused versions are stabler and faster.
2. **Ignore your padding** with `ignore_index=PAD_IDX`, or the model spends its
   capacity learning to predict `<PAD>`.
3. **Class imbalance is a likelihood problem.** If 95% of labels are negative,
   always predicting negative really is the maximum-likelihood answer. Reweight to
   change the question.

The cell below demonstrates rule 2, which matters for lessons 09 and 10.

In [ ]:
V, PAD = 10, 0
torch.manual_seed(0)

# A batch of 4 sequences, 6 tokens each, mostly padding after position 2.
targets = torch.tensor([
    [3, 5, PAD, PAD, PAD, PAD],
    [7, 2, 9,   PAD, PAD, PAD],
    [4, 4, PAD, PAD, PAD, PAD],
    [8, 1, 6,   2,   PAD, PAD],
])
logits = torch.randn(4, 6, V)

naive = nn.CrossEntropyLoss()(logits.reshape(-1, V), targets.reshape(-1))
masked = nn.CrossEntropyLoss(ignore_index=PAD)(logits.reshape(-1, V), targets.reshape(-1))

n_real = (targets != PAD).sum().item()
n_total = targets.numel()

print(f'tokens: {n_real} real, {n_total - n_real} padding ({100*(1-n_real/n_total):.0f}% padding)')
print(f'loss without ignore_index : {naive.item():.4f}')
print(f'loss with    ignore_index : {masked.item():.4f}')
print()
print('The first number is mostly a measure of how well the model predicts <PAD>,')
print('which it will learn to do perfectly and which is worth nothing.')

---
### Concept Check: Choosing and Using Losses

1. You are predicting house prices and the dataset contains a few mansions at 50×
   the median. Which loss, and why?
2. You train a classifier with `nn.CrossEntropyLoss` and apply `F.softmax` yourself
   before passing the output. Nothing crashes. What have you actually done?
3. A sequence model's training loss drops to near zero within two epochs and the
   generated output is nonsense. What is the most likely bug?

In [ ]:
# 1.
# 2.
# 3.

---
## Step 19: The Whole Chapter in One Figure

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))

# -- surprisal ------------------------------------------------------------------
p = np.linspace(0.01, 1, 300)
ax[0].plot(p, -np.log(p), color='#6c5ce7', lw=2.2)
ax[0].set_xlabel('probability assigned to the truth')
ax[0].set_ylabel('loss  −log p')
ax[0].set_title('Cross-entropy: unbounded when wrong')
ax[0].grid(alpha=0.3)

# -- regression losses ----------------------------------------------------------
e = np.linspace(-3, 3, 300)
d = 1.0
ax[1].plot(e, e**2, color='#d63031', lw=2, label='MSE')
ax[1].plot(e, np.abs(e), color='#00b894', lw=2, label='MAE')
ax[1].plot(e, np.where(np.abs(e) <= d, 0.5*e**2, d*(np.abs(e) - 0.5*d)),
           color='#6c5ce7', lw=2, ls='--', label='Huber')
ax[1].set_xlabel('error  e'); ax[1].set_ylabel('loss')
ax[1].set_title('Regression losses')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

# -- entropy and cross-entropy --------------------------------------------------
pp = np.linspace(0.02, 0.98, 300)
H = -(pp*np.log(pp) + (1-pp)*np.log(1-pp))
q = 0.25
CE = -(pp*np.log(q) + (1-pp)*np.log(1-q))
ax[2].plot(pp, H, color='#0984e3', lw=2, label='H(p) — irreducible')
ax[2].plot(pp, CE, color='#d63031', lw=2, ls='--', label='H(p,q), q=0.25')
ax[2].fill_between(pp, H, CE, color='#d63031', alpha=0.12)
ax[2].set_ylim(0, 2.2)
ax[2].set_xlabel('true p'); ax[2].set_ylabel('nats')
ax[2].set_title('The shaded gap is KL(p‖q)')
ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## What You Built

*Linear algebra* — dot products as similarity, matrix multiplication as a grid of
dot products, the shape rules, `nn.Linear` reproduced by hand, and the broadcasting
case that fails silently.

*Calculus* — six derivative rules verified numerically, gradient checking, the chain
rule, summing over paths (and the `zero_grad()` bug it causes), and gradient descent
converging and diverging.

*Statistics* — expectation and variance from the definition, the identity between
sigmoid's derivative and a Bernoulli variance, variance propagation through 20 layers
(deriving Xavier and He initialisation and watching the wrong scale saturate a network
before training starts), the dot-product variance behind attention's `sqrt(d_k)`,
standardisation checked against `nn.LayerNorm`, dropout's preserved mean and exploding
variance, and the standard error over seeds.

*Probability and loss* — BCE from the Bernoulli likelihood, softmax with and without
the max-subtraction trick, cross-entropy from the categorical likelihood, the
`ŷ - y` gradient verified analytically and numerically and by autograd, MSE vs MAE
vs Huber under an outlier, and entropy / KL / perplexity.

## Next

[Lesson 01](../01_nn_from_scratch/prompt.ipynb) uses all three at once: it computes
a forward pass with matrix multiplies, a BCE loss, and every gradient in a two-layer
network by applying the chain rule layer by layer. That is backpropagation — and
after this notebook it introduces no new mathematics at all.

---
# Part 5 · Recognising Which Tool Applies

Parts 1–4 gave you the machinery. The harder skill is **recognition** — noticing that
the thing in front of you is a variance question, or a base-rate question, before you
know what to compute.

Four triggers cover most of what you will meet:

| If something is… | it is a… | tool |
|---|---|---|
| **applied repeatedly** | variance question | `Var(aX) = a²Var(X)`, additivity |
| **measured once** | standard-error question | sample mean, `s/√n` |
| **rare** | base-rate question | Bayes, precision/recall |
| **an output choice** | likelihood question | distribution → NLL |

Plus one habit that applies to every number: **compare it to a baseline** (`ln K`, the
majority class, the uniform model). See
[Chapter 00, Part 5](../book/index.html#ch00-which-tool) for guided examples.

The cell below makes each trigger concrete by computing the thing that decides it.

In [ ]:
# Trigger 1 — APPLIED REPEATEDLY -> variance compounds geometrically.
print('1. applied repeatedly')
for sigma_w, label in [(1.0, 'randn (sigma=1)'), (1/np.sqrt(256), '1/sqrt(n)')]:
    var = 1.0
    for _ in range(30):
        var *= 256 * sigma_w ** 2          # Var(z) = n * sigma_w^2 per layer
    print(f'   {label:<18} variance after 30 layers: {var:.3e}')
print('   -> "loss stuck at ln(K)" plus a deep stack is almost always this.')
print()

# Trigger 2 — MEASURED ONCE -> is the gap bigger than the noise?
print('2. measured once')
runs_a = np.array([0.841, 0.828, 0.855])
runs_b = np.array([0.849, 0.836, 0.861])
for name, r in (('A', runs_a), ('B', runs_b)):
    se = r.std(ddof=1) / np.sqrt(len(r))
    print(f'   model {name}: mean {r.mean():.4f}  SE {se:.4f}')
gap = abs(runs_a.mean() - runs_b.mean())
pooled = np.sqrt((runs_a.std(ddof=1)**2 + runs_b.std(ddof=1)**2) / len(runs_a))
print(f'   gap {gap:.4f} vs combined SE {pooled:.4f} '
      f'-> {"REAL" if gap > 2*pooled else "inside the noise"}')
print()

# Trigger 3 — RARE -> accuracy lies; compute precision.
print('3. rare')
prevalence, recall, specificity = 0.01, 0.90, 0.95
tp = prevalence * recall
fp = (1 - prevalence) * (1 - specificity)
acc = prevalence*recall + (1-prevalence)*specificity
print(f'   accuracy  {acc:.3f}   (always-negative baseline: {1-prevalence:.3f})')
print(f'   precision {tp/(tp+fp):.3f}   <- {100*fp/(tp+fp):.0f}% of alerts are wrong')
print()

# Trigger 4 — OUTPUT CHOICE -> the distribution picks the loss.
print('4. an output choice')
for desc, dist, loss, target in [
    ('one real number',        'Gaussian',    'MSELoss',           '(B,1) float'),
    ('one of K classes',       'Categorical', 'CrossEntropyLoss',  '(B,) int64'),
    ('K independent yes/no',   'Bernoulli x K','BCEWithLogitsLoss','(B,K) float'),
]:
    print(f'   {desc:<24} {dist:<14} {loss:<19} {target}')
print()

# The habit — always quote the baseline.
print('baseline for a cross-entropy, so you can tell if a number is good:')
for K in (2, 10, 60, 50000):
    print(f'   K={K:<6} uniform loss = ln(K) = {np.log(K):.3f}   perplexity {K}')

### Drill: name the trigger

Six situations. For each, decide the trigger **before** reading on, then run the cell to
check. The answer you want is the category, not a number — getting the category right is
most of the work.

1. Widening `d_k` from 64 to 256 made the model worse; attention is nearly one-hot.
2. Validation loss is 3.9 on a 60-character vocabulary. Good?
3. Height and shoe size correlate at 0.8, so shoe size is redundant.
4. House-price regression where 2% of homes are mansions. Which loss?
5. Dropout at p=0.9 made training unstable rather than slower.
6. A 99%-sensitive screening test came back positive.

In [ ]:
ANSWERS = {
    1: ('applied repeatedly (variance)',
        'dot-product scores have SD sqrt(d_k); 64->256 doubled the spread and '
        'saturated the softmax. Check the 1/sqrt(d_k) scaling is applied.'),
    2: ('baseline',
        f'uniform over 60 chars is ln(60) = {np.log(60):.2f}; 3.9 is barely below it, '
        'so the model has learned almost nothing.'),
    3: ('related? (correlation, and its limits)',
        'correlation detects LINEAR association only, and correlated features are not '
        'automatically redundant for a model. Test held-out performance without it.'),
    4: ('an output choice (likelihood)',
        'MSE is the Gaussian NLL and this data has heavy tails, so the assumption is '
        'false. Use MAE or Huber; check the residual distribution.'),
    5: ('applied repeatedly (variance)',
        'inverted dropout preserves the mean but scales variance by p/(1-p) = 9x at '
        'p=0.9. That is injected noise, not slowness.'),
    6: ('rare (base rates)',
        'sensitivity alone cannot give P(disease | +). You need prevalence and '
        'specificity; at 0.5% prevalence the answer is about 9%.'),
}

for i in sorted(ANSWERS):
    trigger, why = ANSWERS[i]
    print(f'{i}. {trigger}')
    print(f'   {why}')
    print()

print('If you got the trigger right but not the detail, that is the intended outcome.')
print('The detail is in Parts 1-4; the trigger is what you have to supply yourself.')

### The point of this part

You are not expected to become a statistician here — a fuller treatment deserves its own
book. This is the minimum needed to read the rest of the curriculum critically: to notice
when a loss curve, an ablation, or a benchmark claim is not carrying the weight put on it.

Two habits are worth more than the rest combined:

- **Quote a baseline next to every number.** `ln K` for a cross-entropy, the majority
  class for accuracy. A metric alone means nothing.
- **Ask how many times it was measured.** One run is an anecdote regardless of the gap.